In [ ]:
import pandas as pd
import numpy as np
import os

ROOT_FOLDER = "annotations"
encode_file = f"{ROOT_FOLDER}/GRCh38-cCREs.bed"
encode = pd.read_csv(encode_file, sep="\t", header=None)
encode.rename(columns={0: "chrom", 1: "start", 2: "end", 3: "cCRE_id", 4: "ENCODE_id", 5: "type"}, inplace=True)
os.makedirs("data", exist_ok=True)
encode.to_csv("data/GRCh38-cCREs.csv", index=False)
encode[encode["type"] == "CA-CTCF"].to_csv("data/ctcf_only.csv", index=False)

In [52]:
genes = pd.read_csv("data/primary_genes_all_features.csv")
genes = genes[genes["feature"] == "gene"]
genes.rename(columns={"seqname": "chrom"}, inplace=True)

In [ ]:
ss

In [50]:
permissive = pd.read_csv("data/human_permissive_enhancers_phase_1_and_2.bed", sep="\t", header=None)

In [77]:
import numpy as np
import pandas as pd

# -----------------------------
# Config
# -----------------------------
W   = 1_000_000   # window around TSS (±1 Mb)
BIN = 100_000     # coarse prefilter bin size

# Tiers (right-closed). Keep your labels and build matching edges.
TIER_EDGES  = [0, 100_000, 250_000, 500_000, 1_000_000]                # bp
TIER_LABELS = ["0–100kb", "100–250kb", "250–500kb", "500–1000kb"]      # 4 labels

# -----------------------------
# Helpers
# -----------------------------
def _normalize_chr(s: pd.Series) -> pd.Series:
    s = s.astype(str)
    return s.str.replace(r"^chr", "", regex=True)

def add_bins(df: pd.DataFrame, start_col: str, end_col: str, bin_size: int = BIN) -> pd.DataFrame:
    """Coarse binning for fast prefilter; exact distances still computed later."""
    out = df.copy()
    out["bin_start"] = (out[start_col] // bin_size).astype(np.int64)
    out["bin_end"]   = (out[end_col]   // bin_size).astype(np.int64)
    out["bins"] = [list(range(s, e + 1)) for s, e in zip(out["bin_start"], out["bin_end"])]
    return out

def _make_cut_edges_for_labels(base_edges, include_zero_bin=True):
    """
    Make edges so that len(labels) == len(edges)-1.
    With base_edges=[0,100k,250k,500k,1M] and 4 labels, we need edges[-0.1,100k,250k,500k,1M].
    """
    if include_zero_bin:
        return [-0.1] + base_edges[1:]
    else:
        return base_edges

def _as_categorical(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = out[c].astype("category")
    return out

# -----------------------------
# Inputs: genes, encode (already loaded)
# genes: chrom, start, end, strand, gene_name
# encode: chrom, start, end, cCRE_id, ENCODE_id, type
# -----------------------------
genes  = genes.copy()
encode = encode.copy()

# Normalize chrom naming
genes["chrom"]  = _normalize_chr(genes["chrom"])
encode["chrom"] = _normalize_chr(encode["chrom"])

# Basic dtypes
genes["start"]  = genes["start"].astype(np.int64)
genes["end"]    = genes["end"].astype(np.int64)
encode["start"] = encode["start"].astype(np.int64)
encode["end"]   = encode["end"].astype(np.int64)

# Optional: set some categoricals for perf
genes  = _as_categorical(genes,  ["chrom", "gene_name", "strand"])
encode = _as_categorical(encode, ["chrom", "type"])

# -----------------------------
# Gene TSS and ±W window (strand-aware; DOES NOT omit '-' genes)
# -----------------------------
genes["tss"]       = np.where(genes["strand"].astype(str) == "+", genes["start"], genes["end"]).astype(np.int64)
genes["win_start"] = (genes["tss"] - W).clip(lower=0).astype(np.int64)
genes["win_end"]   = (genes["tss"] + W).astype(np.int64)

# -----------------------------
# Element metadata & ID
# -----------------------------
encode["center"] = ((encode["start"] + encode["end"]) // 2).astype(np.int64)

# Prefer cCRE_id if present and non-empty; else ENCODE_id
cCRE = encode["cCRE_id"]   if "cCRE_id"   in encode.columns else pd.Series("", index=encode.index)
eID  = encode["ENCODE_id"] if "ENCODE_id" in encode.columns else pd.Series("", index=encode.index)
elem_key = np.where(cCRE.astype(str).str.len() > 0, cCRE.astype(str), eID.astype(str))
encode["elem_id"] = pd.Series(elem_key, index=encode.index).astype("category")

# -----------------------------
# Coarse prefilter via bins
# -----------------------------
genes_bins  = add_bins(genes,  "win_start", "win_end", BIN)
encode_bins = add_bins(encode, "start",     "end",     BIN)

gb = genes_bins.explode("bins")
eb = encode_bins.explode("bins")

pref = (
    gb[["gene_name","chrom","win_start","win_end","tss","bins"]]
      .merge(
        eb[["chrom","start","end","center","elem_id","type","bins"]],
        on=["chrom","bins"], how="inner", copy=False
      )
)

# Tighten with real window overlap
pref = pref[(pref["end"] >= pref["win_start"]) & (pref["start"] <= pref["win_end"])].copy()

# -----------------------------
# Exact distance from TSS to element interval
# distance = 0 if TSS in [start, end]; else min(|tss-start|, |tss-end|)
# -----------------------------
tss = pref["tss"].to_numpy(np.int64)
s   = pref["start"].to_numpy(np.int64)
e   = pref["end"].to_numpy(np.int64)

inside = (tss >= s) & (tss <= e)
dist   = np.where(inside, 0, np.minimum(np.abs(tss - s), np.abs(tss - e)))
pref["dist_to_tss"] = dist.astype(np.int64)

# Keep within ±W by exact distance (explicit)
pref = pref[pref["dist_to_tss"] <= W].copy()

# -----------------------------
# Tier labels (fix edges vs labels mismatch)
# -----------------------------
bins_for_cut = _make_cut_edges_for_labels(TIER_EDGES, include_zero_bin=True)
pref["tier"] = pd.cut(
    pref["dist_to_tss"],
    bins=bins_for_cut,
    labels=TIER_LABELS,
    include_lowest=True,
    right=True,
    ordered=True
)

# -----------------------------
# 1) Long table of all gene–element pairs (exact)
# -----------------------------
pair_df = pref[[
    "gene_name","elem_id","type","chrom","start","end","center","tss","dist_to_tss","tier"
]].copy()

# -----------------------------
# 2) Element-focus table (tiered gene lists + min distance + exact "gene:dist")
# -----------------------------
agg_genes = (pair_df
    .groupby(["elem_id","type","chrom","start","end","tier"], observed=True)["gene_name"]
    .apply(lambda s: ",".join(sorted(pd.unique(s))))
    .reset_index()
)

elem_focus = (agg_genes
    .pivot(index=["elem_id","type","chrom","start","end"], columns="tier", values="gene_name")
    .reset_index()
)

# Cast tier columns to object BEFORE fillna to avoid categorical fill errors
tier_cols = [c for c in elem_focus.columns if c in TIER_LABELS]
for c in tier_cols:
    elem_focus[c] = elem_focus[c].astype(object)
elem_focus[tier_cols] = elem_focus[tier_cols].fillna("")

# Min distance to any gene
elem_focus = elem_focus.merge(
    pair_df.groupby("elem_id", observed=True)["dist_to_tss"].min().rename("min_dist_to_any_gene"),
    on="elem_id", how="left"
)

# Exact "gene:dist" compact string (sorted by distance)
_exact = (pair_df
    .sort_values(["elem_id","dist_to_tss","gene_name"])
    .groupby("elem_id", observed=True)
    .apply(lambda d: ",".join(f"{g}:{int(dd)}" for g, dd in zip(d["gene_name"], d["dist_to_tss"])))
    .rename("genes_by_exact_dist")
    .reset_index()
)
elem_focus = elem_focus.merge(_exact, on="elem_id", how="left")

# -----------------------------
# 3) Gene summary table: counts and IDs per (type, tier), wide format
# -----------------------------
# Counts
cnt = (pair_df
       .groupby(["gene_name","type","tier"], observed=True)["elem_id"]
       .nunique()
       .rename("count")
       .reset_index())

cnt_wide = (cnt
    .assign(col=lambda d: d["type"].astype(str) + " | " + d["tier"].astype(str) + " | count")
    .pivot(index="gene_name", columns="col", values="count")
    .reset_index()
)

# IDs
ids = (pair_df
       .groupby(["gene_name","type","tier"], observed=True)["elem_id"]
       .apply(lambda s: ",".join(sorted(pd.unique(s))))
       .rename("ids")
       .reset_index())

ids_wide = (ids
    .assign(col=lambda d: d["type"].astype(str) + " | " + d["tier"].astype(str) + " | ids")
    .pivot(index="gene_name", columns="col", values="ids")
    .reset_index()
)

# Merge, then fill numeric vs string separately (avoid categorical fill errors)
gene_summary = cnt_wide.merge(ids_wide, on="gene_name", how="outer")

num_cols = [c for c in gene_summary.columns if c.endswith(" | count")]
id_cols  = [c for c in gene_summary.columns if c.endswith(" | ids")]

# ensure proper dtypes before filling
for c in num_cols:
    gene_summary[c] = pd.to_numeric(gene_summary[c], errors="coerce")
for c in id_cols:
    gene_summary[c] = gene_summary[c].astype(object)

gene_summary[num_cols] = gene_summary[num_cols].fillna(0).astype(int)
gene_summary[id_cols]  = gene_summary[id_cols].fillna("")

# Order columns
gene_summary = gene_summary[["gene_name"] + [c for c in gene_summary.columns if c != "gene_name"]]

# -----------------------------
# 4) Distance matrix (genes × elements, min exact distance)
# -----------------------------
dist_matrix = (pair_df
               .groupby(["gene_name","elem_id"], observed=True)["dist_to_tss"]
               .min()
               .unstack("elem_id")
               .astype("float64"))

# Optional: order columns by element’s global min distance
elem_order = pair_df.groupby("elem_id", observed=True)["dist_to_tss"].min().sort_values().index
dist_matrix = dist_matrix.reindex(columns=elem_order)

# -----------------------------
# Outputs:
#   pair_df      -> long gene–element pairs with exact distances + tiers
#   elem_focus   -> per-element tiered gene lists (+ min distance, + "gene:dist")
#   gene_summary -> per-gene wide table split by (type, tier) for counts and IDs
#   dist_matrix  -> genes × elements numeric matrix (min exact distance)
# -----------------------------


C:\Users\stavz\AppData\Local\Temp\ipykernel_37512\105093215.py:171: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: ",".join(f"{g}:{int(dd)}" for g, dd in zip(d["gene_name"], d["dist_to_tss"])))


In [ ]:
pair_df.to_csv("data/regulatory_elements_matching/gene_to_elements.csv", index=False)
elem_focus.to_csv("data/regulatory_elements_matching/regulatory_element_focus.csv", index=False)
gene_summary.to_csv("data/regulatory_elements_matching/gene_focus.csv", index=False)
dist_matrix.to_csv("data/regulatory_elements_matching/distance_matrix.csv", index=False)


lncRNA-enhancer matching

In [ ]:
lncRNA_genes = pd.read_csv("data/lncRNA_matching/lncRNAs_with_genes_1MB.csv")


# -----------------------------
# Config
# -----------------------------
W   = 1_000_000   # window around TSS (±1 Mb)
BIN = 100_000     # coarse prefilter bin size

# Tiers (right-closed). Keep your labels and build matching edges.
TIER_EDGES  = [0, 100_000, 250_000, 500_000, 1_000_000]                # bp
TIER_LABELS = ["0–100kb", "100–250kb", "250–500kb", "500–1000kb"]      # 4 labels

# -----------------------------
# Helpers
# -----------------------------
def _normalize_chr(s: pd.Series) -> pd.Series:
    s = s.astype(str)
    return s.str.replace(r"^chr", "", regex=True)

def add_bins(df: pd.DataFrame, start_col: str, end_col: str, bin_size: int = BIN) -> pd.DataFrame:
    """Coarse binning for fast prefilter; exact distances still computed later."""
    out = df.copy()
    out["bin_start"] = (out[start_col] // bin_size).astype(np.int64)
    out["bin_end"]   = (out[end_col]   // bin_size).astype(np.int64)
    out["bins"] = [list(range(s, e + 1)) for s, e in zip(out["bin_start"], out["bin_end"])]
    return out

def _make_cut_edges_for_labels(base_edges, include_zero_bin=True):
    """
    Make edges so that len(labels) == len(edges)-1.
    With base_edges=[0,100k,250k,500k,1M] and 4 labels, we need edges[-0.1,100k,250k,500k,1M].
    """
    if include_zero_bin:
        return [-0.1] + base_edges[1:]
    else:
        return base_edges

def _as_categorical(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = out[c].astype("category")
    return out

# -----------------------------
# Inputs: genes, encode (already loaded)
# genes: chrom, start, end, strand, gene_name
# encode: chrom, start, end, cCRE_id, ENCODE_id, type
# -----------------------------
genes  = genes.copy()
encode = encode.copy()

# Normalize chrom naming
genes["chrom"]  = _normalize_chr(genes["chrom"])
encode["chrom"] = _normalize_chr(encode["chrom"])

# Basic dtypes
genes["start"]  = genes["start"].astype(np.int64)
genes["end"]    = genes["end"].astype(np.int64)
encode["start"] = encode["start"].astype(np.int64)
encode["end"]   = encode["end"].astype(np.int64)

# Optional: set some categoricals for perf
genes  = _as_categorical(genes,  ["chrom", "gene_name", "strand"])
encode = _as_categorical(encode, ["chrom", "type"])

# -----------------------------
# Gene TSS and ±W window (strand-aware; DOES NOT omit '-' genes)
# -----------------------------
genes["tss"]       = np.where(genes["strand"].astype(str) == "+", genes["start"], genes["end"]).astype(np.int64)
genes["win_start"] = (genes["tss"] - W).clip(lower=0).astype(np.int64)
genes["win_end"]   = (genes["tss"] + W).astype(np.int64)

# -----------------------------
# Element metadata & ID
# -----------------------------
encode["center"] = ((encode["start"] + encode["end"]) // 2).astype(np.int64)

# Prefer cCRE_id if present and non-empty; else ENCODE_id
cCRE = encode["cCRE_id"]   if "cCRE_id"   in encode.columns else pd.Series("", index=encode.index)
eID  = encode["ENCODE_id"] if "ENCODE_id" in encode.columns else pd.Series("", index=encode.index)
elem_key = np.where(cCRE.astype(str).str.len() > 0, cCRE.astype(str), eID.astype(str))
encode["elem_id"] = pd.Series(elem_key, index=encode.index).astype("category")

# -----------------------------
# Coarse prefilter via bins
# -----------------------------
genes_bins  = add_bins(genes,  "win_start", "win_end", BIN)
encode_bins = add_bins(encode, "start",     "end",     BIN)

gb = genes_bins.explode("bins")
eb = encode_bins.explode("bins")

pref = (
    gb[["gene_name","chrom","win_start","win_end","tss","bins"]]
      .merge(
        eb[["chrom","start","end","center","elem_id","type","bins"]],
        on=["chrom","bins"], how="inner", copy=False
      )
)

# Tighten with real window overlap
pref = pref[(pref["end"] >= pref["win_start"]) & (pref["start"] <= pref["win_end"])].copy()

# -----------------------------
# Exact distance from TSS to element interval
# distance = 0 if TSS in [start, end]; else min(|tss-start|, |tss-end|)
# -----------------------------
tss = pref["tss"].to_numpy(np.int64)
s   = pref["start"].to_numpy(np.int64)
e   = pref["end"].to_numpy(np.int64)

inside = (tss >= s) & (tss <= e)
dist   = np.where(inside, 0, np.minimum(np.abs(tss - s), np.abs(tss - e)))
pref["dist_to_tss"] = dist.astype(np.int64)

# Keep within ±W by exact distance (explicit)
pref = pref[pref["dist_to_tss"] <= W].copy()

# -----------------------------
# Tier labels (fix edges vs labels mismatch)
# -----------------------------
bins_for_cut = _make_cut_edges_for_labels(TIER_EDGES, include_zero_bin=True)
pref["tier"] = pd.cut(
    pref["dist_to_tss"],
    bins=bins_for_cut,
    labels=TIER_LABELS,
    include_lowest=True,
    right=True,
    ordered=True
)

# -----------------------------
# 1) Long table of all gene–element pairs (exact)
# -----------------------------
pair_df = pref[[
    "gene_name","elem_id","type","chrom","start","end","center","tss","dist_to_tss","tier"
]].copy()

# -----------------------------
# 2) Element-focus table (tiered gene lists + min distance + exact "gene:dist")
# -----------------------------
agg_genes = (pair_df
    .groupby(["elem_id","type","chrom","start","end","tier"], observed=True)["gene_name"]
    .apply(lambda s: ",".join(sorted(pd.unique(s))))
    .reset_index()
)

elem_focus = (agg_genes
    .pivot(index=["elem_id","type","chrom","start","end"], columns="tier", values="gene_name")
    .reset_index()
)

# Cast tier columns to object BEFORE fillna to avoid categorical fill errors
tier_cols = [c for c in elem_focus.columns if c in TIER_LABELS]
for c in tier_cols:
    elem_focus[c] = elem_focus[c].astype(object)
elem_focus[tier_cols] = elem_focus[tier_cols].fillna("")

# Min distance to any gene
elem_focus = elem_focus.merge(
    pair_df.groupby("elem_id", observed=True)["dist_to_tss"].min().rename("min_dist_to_any_gene"),
    on="elem_id", how="left"
)

# Exact "gene:dist" compact string (sorted by distance)
_exact = (pair_df
    .sort_values(["elem_id","dist_to_tss","gene_name"])
    .groupby("elem_id", observed=True)
    .apply(lambda d: ",".join(f"{g}:{int(dd)}" for g, dd in zip(d["gene_name"], d["dist_to_tss"])))
    .rename("genes_by_exact_dist")
    .reset_index()
)
elem_focus = elem_focus.merge(_exact, on="elem_id", how="left")

# -----------------------------
# 3) Gene summary table: counts and IDs per (type, tier), wide format
# -----------------------------
# Counts
cnt = (pair_df
       .groupby(["gene_name","type","tier"], observed=True)["elem_id"]
       .nunique()
       .rename("count")
       .reset_index())

cnt_wide = (cnt
    .assign(col=lambda d: d["type"].astype(str) + " | " + d["tier"].astype(str) + " | count")
    .pivot(index="gene_name", columns="col", values="count")
    .reset_index()
)

# IDs
ids = (pair_df
       .groupby(["gene_name","type","tier"], observed=True)["elem_id"]
       .apply(lambda s: ",".join(sorted(pd.unique(s))))
       .rename("ids")
       .reset_index())

ids_wide = (ids
    .assign(col=lambda d: d["type"].astype(str) + " | " + d["tier"].astype(str) + " | ids")
    .pivot(index="gene_name", columns="col", values="ids")
    .reset_index()
)

# Merge, then fill numeric vs string separately (avoid categorical fill errors)
gene_summary = cnt_wide.merge(ids_wide, on="gene_name", how="outer")

num_cols = [c for c in gene_summary.columns if c.endswith(" | count")]
id_cols  = [c for c in gene_summary.columns if c.endswith(" | ids")]

# ensure proper dtypes before filling
for c in num_cols:
    gene_summary[c] = pd.to_numeric(gene_summary[c], errors="coerce")
for c in id_cols:
    gene_summary[c] = gene_summary[c].astype(object)

gene_summary[num_cols] = gene_summary[num_cols].fillna(0).astype(int)
gene_summary[id_cols]  = gene_summary[id_cols].fillna("")

# Order columns
gene_summary = gene_summary[["gene_name"] + [c for c in gene_summary.columns if c != "gene_name"]]

# -----------------------------
# 4) Distance matrix (genes × elements, min exact distance)
# -----------------------------
dist_matrix = (pair_df
               .groupby(["gene_name","elem_id"], observed=True)["dist_to_tss"]
               .min()
               .unstack("elem_id")
               .astype("float64"))

# Optional: order columns by element’s global min distance
elem_order = pair_df.groupby("elem_id", observed=True)["dist_to_tss"].min().sort_values().index
dist_matrix = dist_matrix.reindex(columns=elem_order)

# -----------------------------
# Outputs:
#   pair_df      -> long gene–element pairs with exact distances + tiers
#   elem_focus   -> per-element tiered gene lists (+ min distance, + "gene:dist")
#   gene_summary -> per-gene wide table split by (type, tier) for counts and IDs
#   dist_matrix  -> genes × elements numeric matrix (min exact distance)
# -----------------------------


C:\Users\stavz\AppData\Local\Temp\ipykernel_37512\105093215.py:171: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: ",".join(f"{g}:{int(dd)}" for g, dd in zip(d["gene_name"], d["dist_to_tss"])))


In [193]:
pair_df.to_csv("data/regulatory_elements_matching/lncRNA/gene_to_elements.csv", index=False)
elem_focus.to_csv("data/regulatory_elements_matching/lncRNA/regulatory_element_focus.csv", index=False)
gene_summary.to_csv("data/regulatory_elements_matching/lncRNA/gene_focus.csv", index=False)
dist_matrix.to_csv("data/regulatory_elements_matching/lncRNA/distance_matrix.csv", index=False)


In [194]:
genes

,chrom,start,end,gene_name,gene_id,num_genes,genes_within_1Mb,strand,tss,win_start,win_end
983,1,64033935,64035151,ENSG00000306464,ENSG00000306464.1,1,['JAK1'],-,64035151,63035151,65035151
984,1,64094379,64171342,ROR1-AS1,ENSG00000223949.9,1,['JAK1'],-,64171342,63171342,65171342
985,1,64173354,64203449,ENSG00000303693,ENSG00000303693.1,1,['JAK1'],-,64203449,63203449,65203449
986,1,64186776,64192685,ENSG00000284928,ENSG00000284928.2,1,['JAK1'],+,64186776,63186776,65186776
987,1,64231248,64245808,ENSG00000300980,ENSG00000300980.1,1,['JAK1'],-,64245808,63245808,65245808
...,...,...,...,...,...,...,...,...,...,...,...
33666,22,42089630,42090028,ENSG00000270083,ENSG00000270083.1,1,['EP300'],-,42090028,41090028,43090028
33667,22,42090931,42137742,NDUFA6-DT,ENSG00000237037.9,1,['EP300'],+,42090931,41090931,43090931
33668,22,42136433,42139927,ENSG00000232710,ENSG00000232710.1,1,['EP300'],-,42139927,41139927,43139927
33669,22,42138060,42139726,ENSG00000281538,ENSG00000281538.1,1,['EP300'],+,42138060,41138060,43138060


In [10]:
cancer_lncs = pd.read_csv(r"C:\Users/stavz\Desktop\masters\data\general_data\significant_lncs_list.txt", sep="\t")



In [12]:
distances = pd.read_csv(r"C:\Users\stavz\Desktop\masters\APM\data\lncRNA_matching\genes_lncRNAs_1MB_distances.csv")

In [13]:
distances

,chrom,gene_name,gene_strand,gene_start,gene_end,lncRNA_name,lncRNA_start,lncRNA_end,lncRNA_strand,min_distance_bp
0,chr1,JAK1,-,64833223,65067754,ENSG00000306464,64033935,64035151,-,798072
1,chr1,JAK1,-,64833223,65067754,ROR1-AS1,64094379,64171342,-,661881
2,chr1,JAK1,-,64833223,65067754,ENSG00000303693,64173354,64203449,-,629774
3,chr1,JAK1,-,64833223,65067754,ENSG00000284928,64186776,64192685,+,640538
4,chr1,JAK1,-,64833223,65067754,ENSG00000300980,64231248,64245808,-,587415
...,...,...,...,...,...,...,...,...,...,...
2554,chr9,PDCD1LG2,+,5510491,5571282,ENSG00000294323,6225292,6328303,-,654010
2555,chr9,PDCD1LG2,+,5510491,5571282,ENSG00000301961,6408334,6414026,-,837052
2556,chr9,PDCD1LG2,+,5510491,5571282,ENSG00000301977,6410931,6413100,+,839649
2557,chr9,PDCD1LG2,+,5510491,5571282,ENSG00000233367,6415145,6449558,-,843863


In [ ]:
lncRNAs_genes[lncRNAs_genes["gene_name"].isin(cancer_lncs["AATBC"])]

,chrom,start,end,gene_name,gene_id,num_genes,genes_within_1Mb,strand
3167,chr2,7922425,8595477,LINC00299,ENSG00000236790.8,1,['ADAM17'],-
13853,chr7,55235965,55255675,ELDR,ENSG00000280890.2,1,['EGFR'],-
13852,chr7,55179750,55212969,EGFR-AS1,ENSG00000224057.3,1,['EGFR'],-
13844,chr7,54556970,54571726,VSTM2A-OT1,ENSG00000224223.1,1,['EGFR'],+
11754,chr6,31053311,31065229,HCG22,ENSG00000228789.11,5,"['HLA-B', 'HLA-C', 'HLA-E', 'MICA', 'MICB']",+
11755,chr6,31173735,31186118,PSORS1C3,ENSG00000204528.4,5,"['HLA-B', 'HLA-C', 'HLA-E', 'MICA', 'MICB']",-
11789,chr6,31999976,32003521,C4A-AS1,ENSG00000233627.2,8,"['HLA-B', 'HLA-C', 'MICA', 'MICB', 'PSMB8', 'P...",-
11793,chr6,32032713,32036258,C4B-AS1,ENSG00000229776.1,8,"['HLA-B', 'HLA-C', 'MICA', 'MICB', 'PSMB8', 'P...",-
11781,chr6,31764310,31765588,SAPCD1-AS1,ENSG00000235663.1,4,"['HLA-B', 'HLA-C', 'MICA', 'MICB']",-
22026,chr12,67989446,68234686,IFNG-AS1,ENSG00000255733.7,1,['IFNG'],+


In [210]:
distances[distances["lncRNA_name"].isin(cancer_lncs["AATBC"])].sort_values(by="min_distance_bp")

,chrom,gene_name,gene_strand,gene_start,gene_end,lncRNA_name,lncRNA_start,lncRNA_end,lncRNA_strand,min_distance_bp
163,chr12,IFNG,-,68154768,68159740,IFNG-AS1,67989446,68234686,+,0
2482,chr7,EGFR,+,55018820,55211628,EGFR-AS1,55179750,55212969,-,0
2483,chr7,EGFR,+,55018820,55211628,ELDR,55235965,55255675,-,24337
1582,chr6,HLA-C,-,31268749,31272130,PSORS1C3,31173735,31186118,-,82631
699,chr17,SOCS3,-,78356778,78360930,DNAH17-AS1,78484882,78503056,+,123952
1874,chr6,TAP2,-,32821833,32838784,HLA-DQB1-AS1,32659880,32660729,+,161104
1654,chr6,HLA-B,-,31353867,31367067,PSORS1C3,31173735,31186118,-,167749
842,chr19,TGFB1,-,41301587,41353961,LINC01480,41530801,41540899,+,176840
1935,chr6,PSMB8,-,32840717,32844679,HLA-DQB1-AS1,32659880,32660729,+,179988
1997,chr6,PSMB9,+,32844136,32860734,HLA-DQB1-AS1,32659880,32660729,+,183407


In [4]:
import pandas as pd
eps = pd.read_csv(r"C:\Users\stavz\Desktop\masters\epitopes\data\strict_filtered.csv")

In [19]:
eps[eps["gene_symbol"].isin(distances["lncRNA_name"])]["gene_symbol"].unique()

array(['C1RL-AS1'], dtype=object)

In [20]:
distances[distances["lncRNA_name"] == "C1RL-AS1"]

,chrom,gene_name,gene_strand,gene_start,gene_end,lncRNA_name,lncRNA_start,lncRNA_end,lncRNA_strand,min_distance_bp
139,chr12,TAPBPL,+,6451690,6466517,C1RL-AS1,7107990,7122849,+,641473
